# Beca 18 RAG Chatbot
Retrieval-Augmented Generation pipeline to answer questions about the Beca 18 regulations.

## Step 0 — Environment setup

In [1]:
%pip install -q pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import importlib.metadata

import pypdf
import tiktoken
import chromadb
import ipywidgets as widgets
from tqdm.notebook import tqdm
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.genai import types
from IPython.display import display, Markdown

load_dotenv(find_dotenv())
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in .env"

client = genai.Client(api_key=GEMINI_API_KEY)

PACKAGES = [
    "pypdf", "tiktoken", "langchain-text-splitters",
    "google-genai", "chromadb", "ipywidgets", "tqdm",
    "python-dotenv", "sentence-transformers",
]
print("Package versions:")
for pkg in PACKAGES:
    try:
        print(f"  {pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg}: not found")

Package versions:
  pypdf: 6.11.0
  tiktoken: 0.12.0
  langchain-text-splitters: 1.1.2
  google-genai: 2.2.0
  chromadb: 1.5.9
  ipywidgets: 8.1.7
  tqdm: 4.67.1
  python-dotenv: 1.1.0
  sentence-transformers: 5.5.0


## Step 1 — Extract text from PDF

In [3]:
PDF_PATH = "../data/beca18_reglamento.pdf"

def extract_text(pdf_path: str) -> str:
    pages = []
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for i, page in enumerate(reader.pages, start=1):
            raw = page.extract_text() or ""
            cleaned = re.sub(r" {2,}", " ", raw)
            cleaned = re.sub(r"(?<!\n)\n(?!\n)", " ", cleaned)
            pages.append(f"[PAGE {i}]\n{cleaned.strip()}")
    return "\n\n".join(pages)

full_text = extract_text(PDF_PATH)
print(f"Total characters : {len(full_text):,}")
print(f"Total words      : {len(full_text.split()):,}")

Total characters : 374,509
Total words      : 55,202


## Step 2 — Tokenisation and chunking

In [4]:
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)
print(f"Total tokens (cl100k_base): {len(tokens):,}")

Total tokens (cl100k_base): 108,737


### Why `chunk_size=800` with `overlap=100`?

The `all-MiniLM-L6-v2` model generates 384-dimensional vectors optimised for semantic similarity at **paragraph scale** — chunks that are too long dilute the signal, while chunks that are too short lose context.

* **800 characters** captures full regulatory articles (the Beca 18 document uses multi-sentence clauses that belong together semantically), keeping embedding quality high while keeping the total chunk count manageable.
* **100 characters of overlap** (~12 % of chunk size) ensures sentences that straddle a boundary appear in both neighbouring chunks, preventing retrieval gaps at split points.
* Because `all-MiniLM-L6-v2` runs entirely locally with no API limits, chunk count does not affect indexing cost.

In [5]:
METADATA = {"document": "beca18_reglamento", "topic": "beca18", "language": "es"}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "],
)

docs = splitter.create_documents([full_text], metadatas=[METADATA])

avg_len = sum(len(d.page_content) for d in docs) / len(docs)
print(f"Total chunks     : {len(docs):,}")
print(f"Avg chunk length : {avg_len:.0f} chars")

Total chunks     : 762
Avg chunk length : 507 chars


## Step 3 — Embedding functions (local, sentence-transformers)

In [6]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_documents(texts):
    return st_model.encode(texts, show_progress_bar=False).tolist()

def embed_query(text):
    return st_model.encode([text], show_progress_bar=False)[0].tolist()

print("Embedding model loaded: all-MiniLM-L6-v2 (384 dims, runs locally, no API limits).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2 (384 dims, runs locally, no API limits).


## Step 4 — ChromaDB collection and indexing

In [7]:
CHROMA_PATH = "../chroma_db_beca18"
COLLECTION_NAME = "beca18_reglamento"
BATCH_SIZE = 100

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"Dropped existing '{COLLECTION_NAME}' collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

texts_all = [d.page_content for d in docs]
metas_all = [d.metadata for d in docs]
ids_all   = [f"chunk_{i}" for i in range(len(docs))]

print(f"Indexing {len(docs)} chunks...")
for start in tqdm(range(0, len(docs), BATCH_SIZE)):
    end = min(start + BATCH_SIZE, len(docs))
    collection.upsert(
        ids=ids_all[start:end],
        documents=texts_all[start:end],
        embeddings=embed_documents(texts_all[start:end]),
        metadatas=metas_all[start:end],
    )

print(f"Total documents stored in ChromaDB: {collection.count():,}")

Dropped existing 'beca18_reglamento' collection.
Indexing 762 chunks...


  0%|          | 0/8 [00:00<?, ?it/s]

Total documents stored in ChromaDB: 762


## Step 5 — Semantic search

In [8]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    results = collection.query(
        query_embeddings=[embed_query(question)],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"text": text, "metadata": meta, "distance": dist}
        for text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


# --- Test ---
sample_q = "¿Cuáles son los requisitos para postular a la Beca 18?"
results = semantic_search(sample_q, k=5)

print(f"Top-3 results for: '{sample_q}'\n")
for i, r in enumerate(results[:3], 1):
    m = re.search(r"\[PAGE (\d+)\]", r["text"])
    page = f" (page {m.group(1)})" if m else ""
    print(f"[{i}] distance={r['distance']:.4f}{page}")
    print(r["text"][:300])
    print()

Top-3 results for: '¿Cuáles son los requisitos para postular a la Beca 18?'

[1] distance=0.3804
.  B:  Condiciones  priorizables  Clasificación  SISFOH  (Solo válido para  Beca 18)  Pobreza extrema  El criterio otorga puntaje adicional a  los postulantes que acrediten pobreza  extrema, según el Sistema de  Focalización de Hogares (SISFOH) de  Organismo de Focalización e  Información Social (OF

[2] distance=0.3881
. Beca 18 Ordinaria;  b. Beca de Formación en Educación Intercultural Bilingüe - Beca EIB;  c. Beca para Adolescentes con Protección Estatal - Beca Protección;  d. Beca para Comunidades Nativas Amazónicas – Beca CNA;  e. Beca para Licenciados del Servicio Militar Voluntario - Beca FF.AA.;  f. Beca p

[3] distance=0.3910
31    f. Beca para pobladores del Valle de los ríos Apurímac, Ene y Mantaro - Beca  VRAEM;  g. Beca para pobladores residentes en el Huallaga - Beca Huallaga;  h. Beca para Pueblo Afroperuano - Beca PA;  i. Beca para Víctimas de la violencia habida en el país

In [9]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(r"C:\Users\Carlos\Documents\GitHub\beca18-rag-chatbot\.env", override=True)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("Key loaded:", GEMINI_API_KEY[:10], "...")

Key loaded: AIzaSyCcQ4 ...


## Step 6 — Answer generation with Gemini 2.5 Flash

In [14]:
GEN_MODEL = "gemini-1.5-flash"
print("Model changed to:", GEN_MODEL)

Model changed to: gemini-1.5-flash


In [15]:
GEN_MODEL = "gemini-1.5-flash"

SYSTEM_PROMPT = """Eres un asistente experto en el reglamento de la Beca 18 del gobierno peruano.
Responde EXCLUSIVAMENTE con base en los fragmentos de contexto proporcionados.
Cita los números de página usando el formato [PAGE N] cuando estén disponibles en el contexto.
Si el contexto no contiene información suficiente para responder, di exactamente:
    'El documento no contiene información sobre este tema.'
No inventes información ni uses conocimiento externo."""


def answer_with_context(question: str, k: int = 5) -> dict:
    chunks = semantic_search(question, k=k)
    context_block = "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\n{c['text']}" for i, c in enumerate(chunks)
    )
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"Contexto recuperado del reglamento:\n\n{context_block}\n\nPregunta: {question}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
        ),
    )
    return {"answer": response.text, "sources": chunks}


# --- On-topic tests ---
on_topic_questions = [
    "¿Cuáles son los requisitos de elegibilidad para postular a la Beca 18?",
    "¿Cuáles son las modalidades de la Beca 18?",
    "¿A cuánto asciende la subvención mensual o estipendio de la Beca 18?",
    "¿Cuáles son las obligaciones del becario durante sus estudios?",
    "¿Bajo qué condiciones se puede perder la Beca 18?",
]

for q in on_topic_questions:
    try:
        result = answer_with_context(q)
        display(Markdown(f"**Q:** {q}\n\n**A:** {result['answer']}\n\n---"))
    except Exception:
        display(Markdown(f"**Q:** {q}\n\n**A:** Error: API unavailable\n\n---"))

**Q:** ¿Cuáles son los requisitos de elegibilidad para postular a la Beca 18?

**A:** Error: API unavailable

---

**Q:** ¿Cuáles son las modalidades de la Beca 18?

**A:** Error: API unavailable

---

**Q:** ¿A cuánto asciende la subvención mensual o estipendio de la Beca 18?

**A:** Error: API unavailable

---

**Q:** ¿Cuáles son las obligaciones del becario durante sus estudios?

**A:** Error: API unavailable

---

**Q:** ¿Bajo qué condiciones se puede perder la Beca 18?

**A:** Error: API unavailable

---

In [17]:
GEN_MODEL = "gemini-2.0-flash"
print("Model changed to:", GEN_MODEL)

Model changed to: gemini-2.0-flash


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Carlos\Documents\GitHub\beca18-rag-chatbot\.env", override=True)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEN_MODEL = "gemini-2.5-flash"
from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("Key loaded:", GEMINI_API_KEY[:10], "...")

In [19]:
# --- Off-topic test ---
off_topic_q = "¿Cuánto cuesta un pasaje Lima-Cusco en avión?"
result = answer_with_context(off_topic_q)
display(Markdown(f"**Q (off-topic):** {off_topic_q}\n\n**A:** {result['answer']}"))

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 52.653132552s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

## Step 7 — ipywidgets chat interface

In [ ]:
def _page_from_text(text: str) -> str:
    m = re.search(r"\[PAGE (\d+)\]", text)
    return f"Page {m.group(1)}" if m else "Page unknown"


question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre la Beca 18...",
    layout=widgets.Layout(width="70%"),
    description="Pregunta:",
    style={"description_width": "70px"},
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k:",
    style={"description_width": "20px"},
    layout=widgets.Layout(width="250px"),
)
ask_button   = widgets.Button(description="Ask",   button_style="primary", icon="search")
clear_button = widgets.Button(description="Clear", button_style="warning", icon="trash")
output_area  = widgets.Output()


def on_ask(_):
    question = question_input.value.strip()
    if not question:
        return
    with output_area:
        output_area.clear_output(wait=True)
        print("Thinking...")
        try:
            result = answer_with_context(question, k=k_slider.value)
        except Exception as e:
            output_area.clear_output(wait=True)
            print(f"Error: {e}")
            return
        output_area.clear_output(wait=True)
        display(Markdown(f"### Respuesta\n{result['answer']}"))
        source_items = []
        for i, src in enumerate(result["sources"], 1):
            title = f"Fragment {i} — {_page_from_text(src['text'])} (distance {src['distance']:.4f})"
            item_output = widgets.Output()
            with item_output:
                print(src["text"][:500])
            acc = widgets.Accordion(children=[item_output])
            acc.set_title(0, title)
            acc.selected_index = None
            source_items.append(acc)
        display(widgets.VBox([widgets.HTML("<b>Source fragments</b>")] + source_items))


def on_clear(_):
    question_input.value = ""
    output_area.clear_output()


ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

display(widgets.VBox([
    widgets.HBox([question_input, k_slider, ask_button, clear_button]),
    output_area,
]))